TEST SETUP

In [ ]:
from google.colab import drive #Mount drive, if needed
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy.stats import norm
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

def calculate_metrics(krama_correct_array, baseline_correct_array=None, baseline_accuracy=None):
    n = len(krama_correct_array)
    krama_successes = sum(krama_correct_array)
    krama_acc = krama_successes / n

    #95% CI
    ci_low, ci_high = proportion_confint(krama_successes, n, alpha=0.05, method='wilson')

    p_value_str = "N/A"
    test_used = ""

    #McNemar's Test
    if baseline_correct_array is not None:
        table = pd.crosstab(baseline_correct_array, krama_correct_array)
        table = table.reindex(index=[False, True], columns=[False, True], fill_value=0)

        result = mcnemar(table, exact=False, correction=True)
        p_value = result.pvalue
        test_used = "McNemar (Paired)"

        if p_value == 0.0:
            p_value_str = "< 1.0000e-300"
        else:
            p_value_str = "{:.4e}".format(p_value)

    #One-Sample Proportion Z Test
    elif baseline_accuracy is not None and baseline_accuracy > 0:
        stat, p_value = proportions_ztest(count=krama_successes, nobs=n, value=baseline_accuracy)
        test_used = "1-Sample Z-Test"

        if p_value == 0.0 or np.isnan(p_value):
            p0 = baseline_accuracy
            z_stat = (krama_acc - p0) / np.sqrt((p0 * (1 - p0)) / n)
            log_p = norm.logsf(abs(z_stat))
            log10_p = log_p / np.log(10)
            log10_p += np.log10(2)
            p_value_str = f"1.0000e{int(log10_p)}"
        else:
            p_value_str = "{:.4e}".format(p_value)

    return {
        "n": n,
        "Accuracy": round(krama_acc * 100, 2),
        "95% CI": f"[{round(ci_low*100, 2)}, {round(ci_high*100, 2)}]",
        "p-value": p_value_str,
        "Test Used": test_used
    }

VALIDATION SPLIT

In [ ]:
#Choose your desired language
#Choose your particular model

PATH_KRAMA_VAL = 'final_output_file_path_for_model_of_language'
PATH_BASE_VAL = 'final_baseline_file_path_for_same_model_of_same_language'

target_map = {
    'option1': 'A',
    'option2': 'B',
    'option3': 'C',
    'option4': 'D'
}

try:
    krama_val_df = pd.read_csv(PATH_KRAMA_VAL)
    base_val_df = pd.read_csv(PATH_BASE_VAL)

    krama_val_df['target_clean'] = krama_val_df['target'].map(target_map)

    krama_correct_val = (krama_val_df['final_answer'] == krama_val_df['target_clean'])
    base_correct_val = (base_val_df['llama_baseline_pred'] == base_val_df['target_clean'])

    val_results = calculate_metrics(krama_correct_val, baseline_correct_array=base_correct_val)
    print("--- VALIDATION SPLIT ---")
    print("KRAMA vs Baseline:", val_results)

    krama_tier3_correct = (krama_val_df['ensemble_pred'] == krama_val_df['target_clean'])
    ablation_val_results = calculate_metrics(krama_correct_val, baseline_correct_array=krama_tier3_correct)
    print("Ablation (Tier 4 vs Tier 3):", ablation_val_results)

except FileNotFoundError:
    print("Validation files not found. Please update PATH_KRAMA_VAL and PATH_BASE_VAL.")

In [ ]:
#Code segment to print Tier 3 Accuracy of the same model for same language (Validation Split)
krama_val_df = pd.read_csv(PATH_KRAMA_VAL)
krama_val_df['target_clean'] = krama_val_df['target'].map(target_map)
tier3_correct = (krama_val_df['ensemble_pred'] == krama_val_df['target_clean'])
print(f"Tier 3 Accuracy: {round(tier3_correct.mean() * 100, 2)}%")

#DISCLAIMER: Make sure to run the code cell of the particular model of the particular language right before running this segment

TEST SPLIT

In [ ]:
#Choose your desired language
#Choose your particular model

PATH_KRAMA_TEST = 'final_output_file_path_for_model_of_language'
MILU_BASELINE_ACCURACY = 0.00  #Replace with exact decimal accuracy of the particular model of the particular language

try:
    krama_test_df = pd.read_csv(PATH_KRAMA_TEST)
    krama_correct_test = krama_test_df['Final_Correct'].astype(bool)

    test_results = calculate_metrics(krama_correct_test, baseline_accuracy=MILU_BASELINE_ACCURACY)
    print("\n--- TEST SPLIT ---")
    print("KRAMA vs Published Baseline:", test_results)

    tier3_test_correct = krama_test_df['Is_Correct'].astype(bool)
    ablation_test_results = calculate_metrics(krama_correct_test, baseline_correct_array=tier3_test_correct)
    print("Ablation (Tier 4 vs Tier 3):", ablation_test_results)

except FileNotFoundError:
    print("Test file not found. Please update PATH_KRAMA_TEST.")

In [ ]:
#Code segment to print Tier 3 Accuracy of the same model for same language (Test Split)
tier3_accuracy = krama_test_df['Is_Correct'].mean() * 100
print(f"Tier 3 (Ensemble) Accuracy: {tier3_accuracy:.2f}%")

#DISCLAIMER: Make sure to run the code cell of the particular model of the particular language right before running this segment

Sample Output for Validation

In [ ]:
--- VALIDATION SPLIT ---
KRAMA vs Baseline: {'n': 812, 'Accuracy': 47.78, '95% CI': '[44.37, 51.22]', 'p-value': '7.3323e-06', 'Test Used': 'McNemar (Paired)'}
Ablation (Tier 4 vs Tier 3): {'n': 812, 'Accuracy': 47.78, '95% CI': '[44.37, 51.22]', 'p-value': '5.0499e-01', 'Test Used': 'McNemar (Paired)'}

#1st line shows the statistical significance between KRAMA vs Baseline accuracy for the particular model of particular language
#2nd line shows the statistical significance difference between the performances of Tier 3 and Tier 4

Sample Output for Test

In [ ]:
--- TEST SPLIT ---
KRAMA vs Published Baseline: {'n': 4525, 'Accuracy': 40.27, '95% CI': '[38.85, 41.7]', 'p-value': '9.8171e-80', 'Test Used': '1-Sample Z-Test'}
Ablation (Tier 4 vs Tier 3): {'n': 4525, 'Accuracy': 40.27, '95% CI': '[38.85, 41.7]', 'p-value': '2.3649e-15', 'Test Used': 'McNemar (Paired)'}

#1st line shows the statistical significance between KRAMA vs Baseline accuracy for the particular model of particular language
#2nd line shows the statistical significance difference between the performances of Tier 3 and Tier 4